In [4]:
import os
import pickle
import warnings
import numpy as np
from math import factorial

import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

GREEN  = "\033[92m"; BOLD = "\033[1m"
YELLOW = "\033[93m"; RESET = "\033[0m"

class ZPredictor(nn.Module):
    def __init__(self, in_dim=5, out_dim=2, dropout=0.2):
        super().__init__()
        def block(a, b):
            return nn.Sequential(nn.Linear(a, b), nn.BatchNorm1d(b),
                                 nn.ReLU(), nn.Dropout(dropout))
        self.net = nn.Sequential(
            block(in_dim, 64), block(64, 128),
            block(128, 128),   block(128, 64),
            nn.Linear(64, out_dim),
        )
    def forward(self, x):
        return self.net(x)

In [5]:
class HierarchicalGradientPipeline:
    """
    Parameters (fixed constants):
    lambda2  : worker rate          (default 15,700)
    b        : comm. coefficient    (default 6.4e-5)
    D        : data size            (default 60 000)
    P        : param size           (default 784)
    n_iter   : convergence iters    (default 20)
    n_mc     : MC samples per arch  (default 500)
    mc_seed  : RNG seed             (default 42)
    """

    def __init__(self,
                 lambda2=15700, b=6.4e-5, D=60000, P=784,
                 n_iter=20, n_mc=500, mc_seed=42):
        self.lambda2  = lambda2
        self.b        = b
        self.D        = D
        self.P        = P
        self.n_iter   = n_iter
        self.n_mc     = n_mc
        self.mc_seed  = mc_seed

        self._model    = None   
        self._scaler_X = None   
        self._scaler_y = None

    def save(self, path: str):
        """Pickle the entire pipeline (model weights included)."""
        with open(path, "wb") as f:
            pickle.dump(self, f)
        print(f"  Pipeline saved → {path}")

    @staticmethod
    def load(path: str) -> "HierarchicalGradientPipeline":
        """Load a previously saved pipeline."""
        with open(path, "rb") as f:
            obj = pickle.load(f)
        print(f"  Pipeline loaded ← {path}")
        return obj

    def load_zpredictor(self, ckpt_path: str):
        ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)

        self._model = ZPredictor(in_dim=5, out_dim=2)
        self._model.load_state_dict(ckpt["model_state"])
        self._model.eval()

        self._scaler_X = StandardScaler()
        self._scaler_X.mean_  = ckpt["scaler_X_mean"]
        self._scaler_X.scale_ = ckpt["scaler_X_std"]
        self._scaler_X.var_   = ckpt["scaler_X_std"] ** 2
        self._scaler_X.n_features_in_ = 5

        self._scaler_y = StandardScaler()
        self._scaler_y.mean_  = ckpt["scaler_y_mean"]
        self._scaler_y.scale_ = ckpt["scaler_y_std"]
        self._scaler_y.var_   = ckpt["scaler_y_std"] ** 2
        self._scaler_y.n_features_in_ = 2

        print(f"  ZPredictor loaded from: {ckpt_path}")
        return self 

    def _predict_cfit_mufit(self, lambda1, a, r, n2, alpha):
        x = np.array([[lambda1, a, r, n2, alpha]], dtype=np.float32)
        xs = self._scaler_X.transform(x)
        with torch.no_grad():
            ps = self._model(torch.tensor(xs)).numpy()
        pred = self._scaler_y.inverse_transform(ps)
        return float(pred[0, 0]), float(pred[0, 1])

    # mean matching convergence
    def _run_convergence(self, lambda1, a, n2, alpha, n1):
        A = (alpha * n1 + 1) / n1
        B = (alpha * n2 + 1) / n2
        mean_denom = self.D * (a + 1.0 / lambda1)
        s = max(int(alpha * n2), 1)
        r = 1.0 / ((n2 / (s + 1)) ** 2 + (n2 / (s + 1)))

        for _ in range(self.n_iter):
            c_fit, mu_fit = self._predict_cfit_mufit(lambda1, a, r, n2, alpha)
            mean_Z2 = c_fit + 1.0 / mu_fit
            beta    = mean_Z2 / mean_denom - r
            r       = max(B * (A - beta) / (1 + B), 1e-6)

        c_fit, mu_fit = self._predict_cfit_mufit(lambda1, a, r, n2, alpha)
        beta_final    = (c_fit + 1.0 / mu_fit) / mean_denom - r
        return r, beta_final

    # ── internal: single MC iteration (parallel edge) ────────────
    def _simulate_one(self, lambda1, a, n2, alpha, n1, r, beta, rng):
        s2 = max(int(alpha * n2), 1); k2 = n2 - s2
        s1 = max(int(alpha * n1), 1); k1 = n1 - s1

        mu1    = lambda1 / (r * self.D)
        mu2    = self.lambda2 / self.P
        r1     = r + beta
        mu1_l1 = lambda1 / (r1 * self.D) if r1 > 0 else mu1
        c_w2       = a * r  * self.D + self.b * self.P
        c_w1_extra = a * r1 * self.D + self.b * self.P

        wt = (c_w2+ rng.exponential(1.0 / mu1, (n1, n2))+ rng.exponential(1.0 / mu2, (n1, n2)))
        wt.sort(axis=1)
        Z2 = wt[:, k2 - 1]

        L1 = (c_w1_extra
              + rng.exponential(1.0 / mu1_l1, n1)
              + rng.exponential(1.0 / mu2,    n1))

        W1 = np.maximum(Z2, L1)     # parallel: W1 = max(Z2, L1_extra)
        W1.sort()
        return float(W1[k1 - 1])

    #average 
    def _mc_average(self, lambda1, a, n2, alpha, n1, r, beta):
        rng   = np.random.default_rng(self.mc_seed)
        times = [self._simulate_one(lambda1, a, n2, alpha, n1, r, beta, rng)
                 for _ in range(self.n_mc)]
        return float(np.mean(times))

    # valid-pair filter 
    def _valid_pairs(self, N_total, alpha):
        pairs = []
        for n2 in range(2, N_total):
            if N_total % (n2 + 1) != 0:
                continue
            n1 = N_total // (n2 + 1)
            if n1 < 2 or n2 < 2:
                continue
            k2 = n2 - max(int(alpha * n2), 1)
            k1 = n1 - max(int(alpha * n1), 1)
            if k1 >= 1 and k2 >= 1:
                pairs.append((n1, n2))
        return pairs

    def find_best_architecture(self, N_total, lambda1, a, alpha):
        """
        Search all valid (n1,n2) for the given problem spec.
        Returns a dict with the best architecture details.
        """
        if self._model is None:
            raise RuntimeError(
                "No ZPredictor loaded. Call load_zpredictor(ckpt_path) first.")

        print("=" * 60)
        print("  HIERARCHICAL DISTRIBUTED GRADIENT CODING")
        print("  Parallel edge model  |  Mean-matching beta")
        print("=" * 60)
        print(f"  N_total={N_total}  λ1={lambda1}  a={a}  α={alpha}")
        print("=" * 60)

        pairs = self._valid_pairs(N_total, alpha)
        if not pairs:
            print("  No valid (n1,n2) pairs found.")
            return None

        print(f"\n  Valid pairs : {pairs}")
        print(f"  MC samples  : {self.n_mc} per pair\n")

        results = []
        for n1, n2 in pairs:
            k2 = n2 - max(int(alpha * n2), 1)
            k1 = n1 - max(int(alpha * n1), 1)
            try:
                print(f"  Running  (n1={n1}, n2={n2})...", end=" ", flush=True)
                r, beta = self._run_convergence(lambda1, a, n2, alpha, n1)
                mc_t    = self._mc_average(lambda1, a, n2, alpha, n1, r, beta)
                print("done.")
                results.append(dict(n1=n1, n2=n2, k1=k1, k2=k2,
                                    r=r, beta=beta, mc=mc_t))
            except Exception as e:
                print(f"skipped ({e})")

        if not results:
            print("\n  No valid results.")
            return None

        best_idx = min(range(len(results)), key=lambda i: results[i]["mc"])

        col = f"E[T]({self.n_mc})"
        print(f"\n  {'Case (n1, n2)':<20} {col:<20}  Note")
        print("  " + "─" * 56)
        for i, res in enumerate(results):
            label  = f"(n1={res['n1']}, n2={res['n2']})"
            mc_str = f"{res['mc']:.6f} s"
            if i == best_idx:
                print(f"  {GREEN}{BOLD}{label:<20} {mc_str:<20}  ← MINIMUM{RESET}")
            else:
                print(f"  {label:<20} {mc_str:<20}")
        print("  " + "─" * 56)

        best = results[best_idx]
        print(f"\n  {YELLOW}{BOLD}Best  →  n1={best['n1']},  n2={best['n2']}{RESET}")
        print(f"  N = n1×(n2+1) = {best['n1']*(best['n2']+1)}")
        print(f"  k1={best['k1']}  k2={best['k2']}")
        print(f"  r* = {best['r']:.6f}   β* = {best['beta']:.6f}")
        print(f"  E[T_iter]({self.n_mc}) = {best['mc']:.6f} s\n")
        return best

    def __repr__(self):
        loaded = "loaded" if self._model is not None else "not loaded"
        return (f"HierarchicalGradientPipeline("
                f"n_mc={self.n_mc}, n_iter={self.n_iter}, "
                f"ZPredictor={loaded})")

if __name__ == "__main__":
    CKPT = "/Users/varshithareddy/Desktop/z_predictor.pt"
    PKL  = "/Users/varshithareddy/Desktop/pipeline.pkl"
    pipeline = (HierarchicalGradientPipeline(n_mc=500)
                .load_zpredictor(CKPT))

  ZPredictor loaded from: /Users/varshithareddy/Desktop/z_predictor.pt


In [7]:
pipeline2 = HierarchicalGradientPipeline.load(PKL)
best2 = pipeline2.find_best_architecture(N_total=24, lambda1=83000, a=1.2e-5, alpha=0.2)

  Pipeline loaded ← /Users/varshithareddy/Desktop/pipeline.pkl
  HIERARCHICAL DISTRIBUTED GRADIENT CODING
  Parallel edge model  |  Mean-matching beta
  N_total=24  λ1=83000  a=1.2e-05  α=0.2

  Valid pairs : [(8, 2), (6, 3), (4, 5), (3, 7), (2, 11)]
  MC samples  : 500 per pair

  Running  (n1=8, n2=2)... done.
  Running  (n1=6, n2=3)... done.
  Running  (n1=4, n2=5)... done.
  Running  (n1=3, n2=7)... done.
  Running  (n1=2, n2=11)... done.

  Case (n1, n2)        E[T](500)             Note
  ────────────────────────────────────────────────────────
  (n1=8, n2=2)         0.449077 s          
  (n1=6, n2=3)         0.421260 s          
  (n1=4, n2=5)         0.390605 s          
  (n1=3, n2=7)         0.385705 s          
  (n1=2, n2=11)        0.376727 s            ← MINIMUM
  ────────────────────────────────────────────────────────

  Best  →  n1=2,  n2=11
  N = n1×(n2+1) = 24
  k1=1  k2=9
  r* = 0.151108   β* = 0.029457
  E[T_iter](500) = 0.376727 s

